# California Housing — Linear Regression and Regularisation

## Project 4

### Objective
Build a complete regression workflow to predict `MedHouseVal` and compare:

- Ordinary Linear Regression
- Ridge Regression (L2 regularisation)
- Lasso Regression (L1 regularisation)
- Elastic Net (L1 + L2 regularisation)

The project includes EDA, preprocessing, scaling, model evaluation, regularisation
hyperparameter tuning, coefficient analysis, and result exports.

### Dataset note

The standard California Housing dataset has **20,640 observations**, **8 numeric predictive
features**, and the target `MedHouseVal`. It was derived from 1990 U.S. Census data.
The target is expressed in units of $100,000.

This ZIP contains a reproducible 20,640-row practice CSV using the same schema because the
execution environment could not directly download the external dataset. The notebook also
contains a cell showing how to load the official dataset with scikit-learn when internet/data
cache access is available.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

BASE = Path.cwd()
if not (BASE / "dataset").exists():
    BASE = BASE.parent

DATA = BASE / "dataset" / "california_housing.csv"
OUT = BASE / "outputs"
OUT.mkdir(exist_ok=True)

df = pd.read_csv(DATA)
display(df.head())
print("Shape:", df.shape)

## 1. Dataset Information and Data Quality

In [ ]:
display(df.info())
display(df.describe().T)

print("Missing cells:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

## 2. Exploratory Data Analysis — Target Distribution

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df["MedHouseVal"], bins=40)
plt.title("California Housing Target Distribution")
plt.xlabel("MedHouseVal ($100,000 units)")
plt.ylabel("Frequency")
plt.show()

## 3. Median Income vs House Value

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df["MedInc"], df["MedHouseVal"], s=8, alpha=0.35)
plt.title("Median Income vs Median House Value")
plt.xlabel("MedInc")
plt.ylabel("MedHouseVal")
plt.show()

## 4. Geographic EDA

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df["Latitude"], df["MedHouseVal"], s=8, alpha=0.25)
plt.title("Latitude vs Median House Value")
plt.xlabel("Latitude")
plt.ylabel("MedHouseVal")
plt.show()

## 5. Correlation Matrix

In [ ]:
features = [c for c in df.columns if c != "MedHouseVal"]
corr = df[features + ["MedHouseVal"]].corr()
display(corr)
corr.to_csv(OUT / "correlation_matrix.csv")

## 6. Train/Test Split

In [ ]:
X = df[features]
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 7. Preprocessing Pipeline

Standardisation is important for regularised linear models because Ridge, Lasso and Elastic
Net penalise coefficient magnitudes. Putting imputation and scaling inside a `Pipeline`
also prevents preprocessing leakage across cross-validation folds.

In [ ]:
def make_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model)
    ])

## 8. Linear Regression Baseline

In [ ]:
linear = make_pipeline(LinearRegression())
linear.fit(X_train, y_train)
linear_pred = linear.predict(X_test)

linear_metrics = {
    "MAE": mean_absolute_error(y_test, linear_pred),
    "MSE": mean_squared_error(y_test, linear_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, linear_pred)),
    "R2": r2_score(y_test, linear_pred)
}
display(pd.DataFrame([linear_metrics]))

## 9. Ridge Regression — L2 Regularisation

In [ ]:
ridge = make_pipeline(Ridge(alpha=1.0))
ridge.fit(X_train, y_train)
ridge_pred = ridge.predict(X_test)

display(pd.DataFrame([{
    "MAE": mean_absolute_error(y_test, ridge_pred),
    "MSE": mean_squared_error(y_test, ridge_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, ridge_pred)),
    "R2": r2_score(y_test, ridge_pred)
}]))

## 10. Lasso Regression — L1 Regularisation

In [ ]:
lasso = make_pipeline(Lasso(alpha=0.001, max_iter=30000))
lasso.fit(X_train, y_train)
lasso_pred = lasso.predict(X_test)

display(pd.DataFrame([{
    "MAE": mean_absolute_error(y_test, lasso_pred),
    "MSE": mean_squared_error(y_test, lasso_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, lasso_pred)),
    "R2": r2_score(y_test, lasso_pred)
}]))

## 11. Elastic Net — L1 + L2 Regularisation

In [ ]:
elastic = make_pipeline(
    ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=30000)
)
elastic.fit(X_train, y_train)
elastic_pred = elastic.predict(X_test)

display(pd.DataFrame([{
    "MAE": mean_absolute_error(y_test, elastic_pred),
    "MSE": mean_squared_error(y_test, elastic_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, elastic_pred)),
    "R2": r2_score(y_test, elastic_pred)
}]))

## 12. Model Comparison

In [ ]:
models = {
    "Linear Regression": linear,
    "Ridge": ridge,
    "Lasso": lasso,
    "ElasticNet": elastic
}

rows = []
predictions = pd.DataFrame(index=y_test.index)
predictions["Actual"] = y_test

for name, model in models.items():
    pred = model.predict(X_test)
    predictions[name] = pred
    rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "MSE": mean_squared_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred)
    })

comparison = pd.DataFrame(rows)
display(comparison)
comparison.to_csv(OUT / "model_comparison.csv", index=False)
predictions.sort_index().to_csv(OUT / "test_predictions.csv")

## 13. Ridge Alpha Tuning with 5-Fold Cross-Validation

In [ ]:
alphas = np.logspace(-4, 3, 15)

ridge_search = GridSearchCV(
    make_pipeline(Ridge()),
    {"model__alpha": alphas},
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1
)
ridge_search.fit(X_train, y_train)

print("Best Ridge parameters:", ridge_search.best_params_)
print("Best CV RMSE:", -ridge_search.best_score_)

plt.figure(figsize=(8,5))
plt.semilogx(
    alphas,
    -ridge_search.cv_results_["mean_test_score"],
    marker="o"
)
plt.title("Ridge CV RMSE vs Alpha")
plt.xlabel("Alpha")
plt.ylabel("CV RMSE")
plt.show()

## 14. Lasso Alpha Tuning

In [ ]:
lasso_search = GridSearchCV(
    make_pipeline(Lasso(max_iter=50000)),
    {"model__alpha": alphas},
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1
)
lasso_search.fit(X_train, y_train)

print("Best Lasso parameters:", lasso_search.best_params_)
print("Best CV RMSE:", -lasso_search.best_score_)

plt.figure(figsize=(8,5))
plt.semilogx(
    alphas,
    -lasso_search.cv_results_["mean_test_score"],
    marker="o"
)
plt.title("Lasso CV RMSE vs Alpha")
plt.xlabel("Alpha")
plt.ylabel("CV RMSE")
plt.show()

## 15. Coefficient Analysis

In [ ]:
coefficient_rows = []

for name, model in {
    "Linear Regression": linear,
    "Ridge": ridge,
    "Lasso": lasso,
    "ElasticNet": elastic
}.items():
    estimator = model.named_steps["model"]
    for feature, coefficient in zip(features, estimator.coef_):
        coefficient_rows.append([name, feature, coefficient])

coefficients = pd.DataFrame(
    coefficient_rows,
    columns=["Model", "Feature", "Coefficient"]
)
display(coefficients)
coefficients.to_csv(OUT / "model_coefficients.csv", index=False)

## 16. Actual vs Predicted

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(y_test, linear_pred, s=8, alpha=0.35)
mn, mx = y_test.min(), y_test.max()
plt.plot([mn, mx], [mn, mx])
plt.title("Actual vs Predicted — Linear Regression")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.show()

## 17. Conclusion

This project demonstrates why regularisation is useful in linear regression:

- **Linear Regression** provides the unregularised baseline.
- **Ridge** applies L2 shrinkage and is useful when correlated predictors should generally remain.
- **Lasso** applies L1 shrinkage and can drive some coefficients toward exactly zero.
- **Elastic Net** combines L1 and L2 penalties.
- Scaling is performed inside the pipeline so regularisation is applied fairly across features.
- Cross-validation is used to select regularisation strength rather than choosing alpha from the test set.

The test metrics in the exported CSV files should be used as the final reported results for this run.

## 18. Official Dataset Option

When internet access is available, the official scikit-learn dataset can be loaded with:

```python
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing(as_frame=True)
df = housing.frame
```

The official dataset contains 20,640 rows and 8 predictive features plus the target. It has no
missing attribute values according to the scikit-learn documentation.